# Guía de familiarización con los datos del HUN

Este notebook es para que **tú** veas los datos reales y entiendas cómo están armados. Cada sección tiene:

- **Qué estamos mirando** y por qué importa.
- Una consulta (casi siempre agregada, para que corra rápido).
- **Qué buscar**: cómo interpretar lo que sale y qué sería raro.
- ✍️ **Anota**: preguntas para que escribas tus conclusiones (en tu cuaderno o en una celda de texto).

### Reglas antes de empezar

1. **VPN conectada** y `.env` lleno. El notebook usa la misma conexión que `hun.extract`.
2. En VS Code, arriba a la derecha, elige el kernel **`.venv`** (Python del entorno del repo).
3. Corre las celdas **en orden**, de arriba hacia abajo (`Shift + Enter`).
4. **Los resultados que ves aquí son datos del HUN.** No los copies a Claude, a ChatGPT ni al grupo de WhatsApp. Lo que sí puedes compartir con Claude: conclusiones en palabras ("la mayoría", "1-4 h"), nombres de conceptos y errores.
5. **Antes de commitear este notebook, borra los outputs**: menú `...` → *Clear All Outputs*, y guarda. El hook bloquea el commit si quedan outputs, así que no hay forma de que se te escape, pero mejor no depender de eso.
6. Si una celda tarda más de 2-3 minutos, interrúmpela (botón ■) y avísame: puede faltar un índice en la tabla.

Te recomiendo leer en paralelo el `CLAUDE.md` del repo (secciones 4 a 6): ahí está lo que ya descubrimos, para que lo compruebes con tus propios ojos.

## 0. Preparación

In [ ]:
import warnings
import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from hun import config
from hun.db import engine
from hun.extract import load, read_sql

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["figure.figsize"] = (11, 4)

ENG = engine()
SQL_DIR = config.REPO_ROOT / "sql"


def q(sql: str) -> pd.DataFrame:
    # Corre una consulta y devuelve un DataFrame (no guarda nada en disco)
    with ENG.connect() as con:
        return pd.read_sql_query(text(sql), con)


def con_episodios(sql_final: str) -> pd.DataFrame:
    # Antepone el bloque de episodios (sql/01_episodios/_episodios.sql) a una consulta
    bloque = read_sql(SQL_DIR / "01_episodios" / "_episodios.sql")
    return q(bloque + "\n" + sql_final)


print(q("select version()").iloc[0, 0].split(",")[0])
print("Esquema:", config.schema())

## 1. Vistazo general: cuánto hay y de cuándo

**Qué estamos mirando:** el tamaño de cada tabla y el rango de fechas. Es lo primero que hay que saber de cualquier base: si una tabla es enorme, las consultas sobre ella hay que pensarlas bien; si las fechas tienen huecos, el análisis temporal se complica.

`reltuples` es el conteo **aproximado** que guarda Postgres (instantáneo); un `COUNT(*)` exacto en tablas de millones tarda.

In [ ]:
q('''
SELECT c.relname AS tabla,
       c.reltuples::bigint AS filas_aprox,
       pg_size_pretty(pg_total_relation_size(c.oid)) AS tamano
FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace
WHERE n.nspname = current_schema() AND c.relkind = 'r'
ORDER BY c.reltuples DESC
''')

**Qué buscar:** si `filas_aprox` sale en 0 o -1, Postgres no ha calculado estadísticas; no es que la tabla esté vacía.

Ahora, **índices**: sin índice en `visit_occurrence_id` o `person_id`, los cruces entre tablas grandes son lentos. Esto explica por qué algunas consultas tardan.

In [ ]:
q('''
SELECT tablename AS tabla, indexname AS indice, indexdef AS definicion
FROM pg_indexes
WHERE schemaname = current_schema()
ORDER BY tablename
''')

In [ ]:
rango = q('''
SELECT 'visit_occurrence' AS tabla, MIN(visit_start_datetime) AS desde, MAX(visit_start_datetime) AS hasta FROM visit_occurrence
UNION ALL SELECT 'visit_detail', MIN(visit_detail_start_datetime), MAX(visit_detail_start_datetime) FROM visit_detail
UNION ALL SELECT 'condition_occurrence', MIN(condition_start_datetime), MAX(condition_start_datetime) FROM condition_occurrence
UNION ALL SELECT 'procedure_occurrence', MIN(procedure_date)::timestamp, MAX(procedure_date)::timestamp FROM procedure_occurrence
UNION ALL SELECT 'measurement', MIN(measurement_datetime), MAX(measurement_datetime) FROM measurement
''')
rango

**Qué buscar:**
- ¿Todas las tablas cubren el mismo periodo? Si una empieza años antes, quizá cambió el sistema de registro.
- ¿Hay fechas absurdas (año 1900, fechas futuras)? Son errores de digitación; anótalos.

✍️ **Anota:** ¿desde y hasta cuándo hay datos útiles?

## 2. Volumen en el tiempo

**Qué estamos mirando:** visitas por mes, por tipo. Así se ven tendencias, estacionalidad y meses incompletos o con caídas raras (cambio de sistema, pandemia, extracción a medias).

In [ ]:
mes = q('''
SELECT date_trunc('month', vo.visit_start_datetime) AS mes, c.concept_name AS tipo, COUNT(*) AS visitas
FROM visit_occurrence vo JOIN concept c ON c.concept_id = vo.visit_concept_id
GROUP BY 1, 2 ORDER BY 1
''')
tabla_mes = mes.pivot(index="mes", columns="tipo", values="visitas").fillna(0)
tabla_mes.plot(title="Visitas por mes según tipo")
plt.ylabel("visitas"); plt.show()
tabla_mes.tail(8)

**Qué buscar:**
- **Los últimos meses**: ya sabemos que vienen incompletos (la base se extrajo a mitad de camino). ¿Desde cuál exactamente cae?
- **Caídas o saltos bruscos** en medio de la serie: ¿coinciden con algo (pandemia, cambio de sistema)? Si un tipo de visita "aparece" de repente, puede ser un cambio en cómo se registra.
- **Estacionalidad**: ¿hay meses que siempre suben (temporada de lluvias, diciembre)?

✍️ **Anota:** ¿qué periodo usarías para entrenar un modelo? ¿Qué meses excluirías?

## 3. Pacientes (`person`)

**Qué estamos mirando:** sexo y edad. Sirve para describir la población y para detectar errores (edades imposibles).

In [ ]:
sexo = q('''
SELECT COALESCE(c.concept_name, 'sin concepto') AS sexo, COUNT(*) AS pacientes
FROM person p LEFT JOIN concept c ON c.concept_id = p.gender_concept_id
GROUP BY 1 ORDER BY 2 DESC
''')
sexo["%"] = 100 * sexo["pacientes"] / sexo["pacientes"].sum()
sexo

In [ ]:
nac = q("SELECT year_of_birth AS anio, COUNT(*) AS pacientes FROM person GROUP BY 1 ORDER BY 1")
nac.plot(x="anio", y="pacientes", kind="bar", legend=False, title="Pacientes por año de nacimiento")
plt.xticks(nac.index[::10], nac["anio"].iloc[::10]); plt.show()
print("Años de nacimiento más antiguos:", nac["anio"].head(5).tolist())
print("Más recientes:", nac["anio"].tail(5).tolist())

**Qué buscar:**
- Nacidos antes de ~1910 → edades de más de 110 años: casi seguro errores.
- Picos raros en un año "redondo" (1900, 2000) suelen ser valores por defecto cuando no se conoce la fecha.
- ¿`race_concept_id` y `ethnicity_concept_id` traen algo, o todo es 0? (Si es todo 0, no sirven.)

In [ ]:
q('''
SELECT 'race' AS campo, race_concept_id AS concept_id, COUNT(*) AS n FROM person GROUP BY 2
UNION ALL
SELECT 'ethnicity', ethnicity_concept_id, COUNT(*) FROM person GROUP BY 2
ORDER BY campo, n DESC
''')

## 4. Visitas y segmentos: cómo se registra el paso por el hospital

**Qué estamos mirando:** esta es la parte más importante. En la exploración vimos que:

| Concepto | Qué parece ser |
|---|---|
| 9203 *Emergency Room Visit* | Marca de llegada a urgencias (dura 0 o minutos) |
| 3045729 *Treatment urgency* | La estancia real en urgencias (horas) |
| 9201 *Inpatient Visit* | Hospitalización (días) |
| 581479 *Ambulatory Rehabilitation Visit* | Eventos ambulatorios puntuales (duran 0) |

Aquí lo compruebas tú. Primero, duración de los segmentos de `visit_detail` por tipo, en horas:

In [ ]:
dur = q('''
SELECT c.concept_name AS segmento,
       COUNT(*) AS n,
       AVG((vd.visit_detail_end_datetime = vd.visit_detail_start_datetime)::int) AS prop_dura_cero,
       PERCENTILE_CONT(ARRAY[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]) WITHIN GROUP (
           ORDER BY EXTRACT(EPOCH FROM (vd.visit_detail_end_datetime - vd.visit_detail_start_datetime)) / 3600
       ) AS percentiles_horas
FROM visit_detail vd JOIN concept c ON c.concept_id = vd.visit_detail_concept_id
GROUP BY 1 ORDER BY n DESC
''')
p = pd.DataFrame(dur["percentiles_horas"].tolist(), columns=["p10", "p25", "p50", "p75", "p90", "p99"])
pd.concat([dur.drop(columns="percentiles_horas"), p], axis=1)

**Qué buscar:**
- ¿*Emergency Room Visit* casi siempre dura 0 o minutos? → es una marca, no una estancia.
- ¿*Treatment urgency* tiene la mediana en horas? → estancia en urgencias.
- ¿*Inpatient* en días? → hospitalización.
- Mira el **p99**: ahí se ven los casos absurdos (meses). ¿Cuántos son?

### Mira un caso real, paso a paso

Nada enseña más que ver la línea de tiempo de un paciente. Esta celda busca, en una muestra pequeña, una visita que pase por urgencias **y** por hospitalización, y muestra todos sus segmentos en orden. Córrela varias veces para ver casos distintos.

In [ ]:
caso = q('''
WITH muestra AS (
  SELECT visit_occurrence_id
  FROM visit_detail TABLESAMPLE SYSTEM (1)
  WHERE visit_detail_concept_id = 9201
  LIMIT 300
), candidata AS (
  SELECT m.visit_occurrence_id
  FROM muestra m
  WHERE EXISTS (SELECT 1 FROM visit_detail d
                WHERE d.visit_occurrence_id = m.visit_occurrence_id
                  AND d.visit_detail_concept_id IN (9203, 3045729))
  ORDER BY random() LIMIT 1
)
SELECT vd.visit_occurrence_id, c.concept_name AS segmento,
       vd.visit_detail_start_datetime AS inicio,
       vd.visit_detail_end_datetime   AS fin,
       ROUND(EXTRACT(EPOCH FROM (vd.visit_detail_end_datetime - vd.visit_detail_start_datetime))::numeric / 3600, 2) AS horas
FROM visit_detail vd
JOIN candidata ca ON ca.visit_occurrence_id = vd.visit_occurrence_id
JOIN concept c ON c.concept_id = vd.visit_detail_concept_id
ORDER BY vd.visit_detail_start_datetime, vd.visit_detail_id
''')
if caso.empty:
    print("No salió ningún caso en esta muestra; vuelve a correr la celda.")
caso

**Qué buscar en cada caso:**
- ¿Primero aparece *Emergency Room Visit* (0 min) y casi al mismo tiempo empieza *Treatment urgency*?
- ¿Hay varios *Treatment urgency* seguidos (el fin de uno = el inicio del siguiente)? Serían cambios de cubículo o de turno.
- ¿Entre el último *Treatment urgency* y el primer *Inpatient* hay hueco? ¿O se solapan (urgencias sigue abierto después de que el paciente ya está en cama)? Ese solape es el **doble conteo** que encontramos.
- ¿La visita incluye segmentos de otros días (por ejemplo, un *Ambulatory* una semana antes)? Por eso una visita no es un episodio.

✍️ **Anota:** describe con tus palabras 2-3 casos. ¿Encajan con la tabla de arriba? ¿Viste algo que no encaja?

## 5. Episodios: el tiempo en urgencias según el destino

**Qué estamos mirando:** usamos el bloque `sql/01_episodios/_episodios.sql`, que une los segmentos seguidos de un mismo paciente (llegada → atención → cama) en **episodios**. Lee `docs/METODOLOGIA_EPISODIOS.md` si quieres el detalle.

El histograma compara el tiempo en urgencias de quienes **se van de alta** contra los que **terminan hospitalizados**. Esta celda reconstruye todos los episodios, así que puede tardar unos minutos.

In [ ]:
hist = con_episodios('''
SELECT destino,
       CASE WHEN min_urg < 15 THEN '0) <15 min'
            WHEN min_urg < 60 THEN '1) 15-60 min'
            WHEN min_urg < 240 THEN '2) 1-4 h'
            WHEN min_urg < 720 THEN '3) 4-12 h'
            WHEN min_urg < 1440 THEN '4) 12-24 h'
            WHEN min_urg < 4320 THEN '5) 1-3 días'
            WHEN min_urg < 10080 THEN '6) 3-7 días'
            ELSE '7) >7 días' END AS franja,
       COUNT(*) AS episodios
FROM epu GROUP BY 1, 2 ORDER BY 1, 2
''')
tabla = hist.pivot(index="franja", columns="destino", values="episodios").fillna(0)
(100 * tabla / tabla.sum()).plot(kind="bar", title="Tiempo en urgencias según destino (% de cada grupo)")
plt.ylabel("% de episodios"); plt.xticks(rotation=30); plt.show()
tabla.assign(**{f"% {c}": 100 * tabla[c] / tabla[c].sum() for c in tabla.columns})

**Qué buscar:**
- Los de **alta**: ¿hay dos grupos (muchos de menos de 15 min y otro de varias horas)? Los de menos de 15 min podrían ser pacientes clasificados y remitidos (triaje IV-V) o solo registros. Pregunta abierta para el HUN.
- Los **hospitalizados**: ¿se corren hacia la derecha (12-24 h, días)? Esa diferencia es la **espera de cama**, nuestro hallazgo principal.
- La cola de **más de 7 días** en urgencias: ¿es creíble o son registros que nadie cerró?

✍️ **Anota:** ¿qué porcentaje de los que llegan a urgencias se hospitaliza? ¿Cuál es la mediana de cada grupo en horas?

In [ ]:
r = load("20_resumen_episodios")      # resultado del paso 4, guardado en tu DATA_DIR
r.assign(**{c.replace("min", "h"): r[c] / 60 for c in r.columns if c.startswith("p") and "min" in c})

## 6. Llegadas: cuándo llega la gente

**Qué estamos mirando:** la serie de llegadas a urgencias por hora (consulta 21) y el tiempo en urgencias según el día y la hora de llegada (consulta 22). Son la base del pronóstico y del heatmap del dashboard.

In [ ]:
lleg = load("21_llegadas_por_hora")
lleg["hora"] = pd.to_datetime(lleg["hora"])
s = lleg.set_index("hora")["llegadas"].asfreq("h", fill_value=0)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
s.resample("W").sum().plot(ax=ax[0], title="Llegadas por semana")
s.groupby(s.index.hour).mean().plot(kind="bar", ax=ax[1], title="Promedio por hora del día")
s.groupby(s.index.dayofweek).mean().plot(kind="bar", ax=ax[2], title="Promedio por día (0 = lunes)")
plt.tight_layout(); plt.show()

**Qué buscar:**
- **Por hora**: ¿cuándo es el pico? ¿Cuántas veces más llega a la hora pico que a la hora valle?
- **Por día**: ¿el lunes es el más cargado (típico en urgencias)?
- **Por semana**: ¿hay estacionalidad a lo largo del año? ¿Dónde se cae la serie al final?

In [ ]:
fr = load("22_tiempo_urgencias_por_franja")
dias = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
for i, destino in enumerate(["alta", "hospitalizado"]):
    m = fr[fr["destino"] == destino].pivot(index="dow", columns="hora", values="p50_min_urg") / 60
    im = ax[i].imshow(m, aspect="auto", cmap="viridis")
    ax[i].set_yticks(range(len(m.index)), [dias[d - 1] for d in m.index])
    ax[i].set_xlabel("hora de llegada"); ax[i].set_title(f"Mediana de horas en urgencias · {destino}")
    plt.colorbar(im, ax=ax[i])
plt.tight_layout(); plt.show()

**Qué buscar:** ¿hay franjas donde quedarse en urgencias dura más? Si los que llegan de noche o el fin de semana esperan más, puede ser por menos personal o porque no se suben pacientes a piso. Eso alimenta las recomendaciones.

✍️ **Anota:** las 2-3 franjas más críticas.

## 7. Ocupación hora a hora

**Qué estamos mirando:** cuántos pacientes hay al mismo tiempo en urgencias y en hospitalización (consulta 25, ya ajustada para no contar dos veces a quien está en urgencias y en cama a la vez).

⚠️ Recuerda el diagnóstico del paso 4b: la ocupación sale **muy por encima** de 58 cubículos y 64 camas. Lo más probable es que esas capacidades no cubran todo lo que hay en la base (otras sedes, UCI, sillas). Mira los **patrones**, no el número absoluto.

In [ ]:
oc = load("25_ocupacion_ajustada")
oc["hora"] = pd.to_datetime(oc["hora"])
oc = oc[oc["hora"] >= oc["hora"].min() + pd.Timedelta(days=30)].set_index("hora")

fig, ax = plt.subplots(1, 2, figsize=(15, 4))
oc.resample("D").mean().plot(ax=ax[0], title="Ocupación promedio diaria")
oc.groupby(oc.index.hour).mean().plot(ax=ax[1], title="Ocupación promedio por hora del día")
plt.tight_layout(); plt.show()
oc.describe()

**Qué buscar:**
- ¿A qué hora está más llena urgencias? ¿Coincide con el pico de llegadas o va unas horas después (porque la gente se acumula)?
- ¿Hospitalización cambia poco durante el día? Las camas se liberan con las altas, que suelen darse a ciertas horas.
- ¿Qué valor típico tiene cada una? Compáralo con 58 y 64: ¿cuántas veces más? Eso es lo que hay que preguntarle al HUN.

## 8. Diagnósticos, procedimientos y laboratorios

**Qué estamos mirando:** estas tablas todavía no las exploramos. Pueden servir para describir **qué** atiende urgencias y, quizá, como sustituto de la gravedad (ya que no hay triaje).

In [ ]:
q('''
SELECT c.concept_name AS diagnostico, c.vocabulary_id, COUNT(*) AS n
FROM condition_occurrence co
JOIN concept c ON c.concept_id = co.condition_concept_id
GROUP BY 1, 2 ORDER BY n DESC LIMIT 25
''')

In [ ]:
q('''
SELECT COALESCE(c.concept_name, 'sin concepto') AS tipo_registro, COUNT(*) AS n
FROM condition_occurrence co
LEFT JOIN concept c ON c.concept_id = co.condition_type_concept_id
GROUP BY 1 ORDER BY n DESC
''')

In [ ]:
q('''
SELECT c.concept_name AS procedimiento, c.vocabulary_id, COUNT(*) AS n
FROM procedure_occurrence po
JOIN concept c ON c.concept_id = po.procedure_concept_id
GROUP BY 1, 2 ORDER BY n DESC LIMIT 25
''')

In [ ]:
q('''
SELECT c.concept_name AS laboratorio, COUNT(*) AS n,
       AVG((m.value_as_number IS NULL)::int) AS prop_sin_valor
FROM measurement m
LEFT JOIN concept c ON c.concept_id = m.measurement_concept_id
GROUP BY 1 ORDER BY n DESC LIMIT 25
''')

**Qué buscar:**
- **Diagnósticos**: ¿qué vocabulario usan (SNOMED, ICD10)? ¿Los más frecuentes tienen sentido para urgencias (dolor abdominal, fiebre, trauma)? ¿Hay muchos con concepto 0 (sin mapear)?
- **Tipo de registro** del diagnóstico: ¿es diagnóstico principal, de ingreso, de egreso? Importa para no contar dos veces.
- **Procedimientos**: ¿qué domina (imágenes, laboratorios, consultas)?
- **Laboratorios**: ¿cuántos no tienen valor numérico? El concepto 0 (sin mapear) era el más frecuente en la exploración.

✍️ **Anota:** ¿alguna de estas tablas te parece útil como "gravedad" del paciente? (Por ejemplo: número de laboratorios por episodio, o ciertos diagnósticos.)

## 9. Chequeo de calidad

Una lista rápida de cosas que suelen estar mal en bases hospitalarias. Cada fila es un problema posible; si la columna `n` sale alta, anótalo para el documento (sección de limitaciones).

In [ ]:
q('''
SELECT 'segmentos con fin antes del inicio' AS chequeo, COUNT(*) AS n
  FROM visit_detail WHERE visit_detail_end_datetime < visit_detail_start_datetime
UNION ALL SELECT 'segmentos sin fecha de fin', COUNT(*)
  FROM visit_detail WHERE visit_detail_end_datetime IS NULL
UNION ALL SELECT 'segmentos con fecha futura', COUNT(*)
  FROM visit_detail WHERE visit_detail_start_datetime > now()
UNION ALL SELECT 'segmentos de más de 90 días', COUNT(*)
  FROM visit_detail WHERE visit_detail_end_datetime - visit_detail_start_datetime > interval '90 days'
UNION ALL SELECT 'segmentos duplicados exactos', COUNT(*) - COUNT(DISTINCT (visit_occurrence_id, visit_detail_concept_id,
                                                         visit_detail_start_datetime, visit_detail_end_datetime))
  FROM visit_detail
UNION ALL SELECT 'visitas sin paciente en person', COUNT(*)
  FROM visit_occurrence vo WHERE NOT EXISTS (SELECT 1 FROM person p WHERE p.person_id = vo.person_id)
UNION ALL SELECT 'visitas sin ningún segmento', COUNT(*)
  FROM visit_occurrence vo WHERE NOT EXISTS (SELECT 1 FROM visit_detail vd WHERE vd.visit_occurrence_id = vo.visit_occurrence_id)
UNION ALL SELECT 'pacientes sin ninguna visita', COUNT(*)
  FROM person p WHERE NOT EXISTS (SELECT 1 FROM visit_occurrence vo WHERE vo.person_id = p.person_id)
''')

**Qué buscar:** cualquier `n` distinto de 0 merece una línea en las limitaciones del documento. Los duplicados exactos, si son muchos, inflan los conteos y hay que quitarlos antes de construir episodios.

---

## 10. Cierre: lo que deberías tener anotado

Al terminar, deberías poder responder en palabras:

1. ¿Qué periodo usamos y qué meses excluimos?
2. ¿Qué es cada concepto de visita/segmento, según lo que viste en los casos reales?
3. ¿Cuánto tiempo pasan en urgencias los que se van de alta y los que se hospitalizan? ¿Cuánta diferencia hay?
4. ¿Cuándo llega más gente y cuándo está más llena urgencias?
5. ¿Qué tan creíble es la ocupación frente a 58 cubículos / 64 camas?
6. ¿Qué problemas de calidad encontraste?
7. ¿Ves algo que contradiga lo que está en el `CLAUDE.md`?

Esas respuestas, **en palabras y sin cifras exactas**, son lo que puedes llevar a la reunión del equipo y, si quieres, compartir conmigo.

> Antes de cerrar: **Clear All Outputs** y guarda. Si vas a commitear el notebook, el hook revisa que no tenga outputs.